In [1]:
import re
import unicodedata
from urllib.request import urlopen, Request
from urllib.error import URLError
from html.parser import HTMLParser
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

OUTPUT_DIR = Path("../data/gutenberg/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"}


## Helpers

In [2]:
def to_snake(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text[:120]

def fetch(url: str) -> str:
    req = Request(url, headers=HEADERS)
    return urlopen(req, timeout=15).read().decode("utf-8", errors="ignore")


## Fetch Spanish catalog

In [3]:
class SpanishCatalogParser(HTMLParser):
    """Extracts (id, title) pairs from gutenberg.org/browse/languages/es."""

    def __init__(self):
        super().__init__()
        self.books: list[tuple[str, str]] = []
        self._current_id: str | None = None
        self._capture = False

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            href = dict(attrs).get("href", "")
            m = re.match(r"^/ebooks/(\d+)$", href)
            if m:
                self._current_id = m.group(1)
                self._capture = True

    def handle_data(self, data):
        if self._capture:
            title = data.strip()
            if title:
                self.books.append((self._current_id, title))
            self._capture = False

    def handle_endtag(self, tag):
        if tag == "a":
            self._capture = False


print("Fetching Spanish catalog...")
html = fetch("https://www.gutenberg.org/browse/languages/es")
parser = SpanishCatalogParser()
parser.feed(html)
books = parser.books
print(f"Found {len(books)} Spanish books")


Fetching Spanish catalog...
Found 1329 Spanish books


## Download books

In [4]:
def build_urls(book_id: str) -> list[str]:
    return [
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
    ]

def clean_text(text: str) -> str:
    start = re.search(r"\*\*\* START OF.*?\*\*\*", text, re.IGNORECASE)
    end   = re.search(r"\*\*\* END OF.*?\*\*\*",   text, re.IGNORECASE)
    if start and end:
        return text[start.end():end.start()].strip()
    return text.strip()

def download(book_id: str, title: str) -> str:
    dest = OUTPUT_DIR / (to_snake(title) + ".txt")
    if dest.exists():
        return f"SKIP {title}"
    for url in build_urls(book_id):
        try:
            data = fetch(url)
            if len(data) > 500:
                dest.write_text(clean_text(data), encoding="utf-8")
                return f"OK   {title}"
        except (URLError, Exception):
            continue
    return f"FAIL {title}"


In [5]:
results = []

with ThreadPoolExecutor(max_workers=10) as ex:
    futures = {ex.submit(download, bid, title): title for bid, title in books}
    for f in as_completed(futures):
        r = f.result()
        print(r)
        results.append(r)

ok   = sum(1 for r in results if r.startswith("OK"))
skip = sum(1 for r in results if r.startswith("SKIP"))
fail = sum(1 for r in results if r.startswith("FAIL"))

print(f"\nOK: {ok}  SKIP: {skip}  FAIL: {fail}")


OK   Historia natural y moral de las Indias (vol. 2 of 2)
OK   Místicas; poesías
OK   Historia de Venezuela, Tomo I
OK   La nariz de un notario
OK   Historia natural y moral de las Indias (vol. 1 of 2)
OK   Germana
OK   Tragedias
OK   Historia de Venezuela, Tomo II
OK   Tratado de Ortografía Valenciana Clásica
OK   Reseña Veridica de la Revolución Filipina
OK   Los Merodeadores de Fronteras
OK   El clavo
OK   Las noches mejicanas
FAIL Las Fábulas de Esopo, Vol. 02
OK   Novelas Cortas
OK   Novelas Cortas
OK   Viajes por España
FAIL Las Fábulas de Esopo, Vol. 03
FAIL Las Fábulas de Esopo, Vol. 01
OK   El Niño de la Bola: Novela
OK   Cosas que fueron: Cuadros de costumbres
OK   Doctor Sutilis (Cuentos)
OK   El Señor y los demás son Cuentos
OK   El tratado de la pintura
OK   La Regenta
OK   Recuerdos de un anciano
FAIL El Capitán Veneno
OK   Nuevas poesías y evangélicas
OK   Bug-Jargal
OK   Viajes por Filipinas: De Manila á Albay
OK   Viajes por Filipinas: De Manila á Tayabas
FAIL El sombr